# Bronze to Silver — VRA/ANAC

Cria `tb_anac_silver` a partir da Bronze (`tb_vra_bronze`), aplicando: tipagem explícita,
dedup via `QUALIFY ROW_NUMBER()`, filtro `REALIZADO`,
descarte de outliers e de pares incompletos e cálculo do
atraso de partida.

**Por que SQL server-side, e não pandas:**
o VRA tem escala bem maior que o dataset de referência (~2 milhões de
linhas no recorte de 3 anos) — trazer tudo para um DataFrame antes de
gravar seria desnecessário e lento. Por isso a transformação inteira roda
como um único `CREATE OR REPLACE TABLE ... AS SELECT`, e o client Python
só dispara o job e valida o resultado.

In [ ]:
from google.cloud import bigquery

## Configuração dos parâmetros

In [ ]:
PROJECT_ID = "pdm-bia-2026"
DATASET_ID = "tf_anac"
BRONZE_TABLE = "tb_vra_bronze"
SILVER_TABLE = "tb_anac_silver"

FULL_BRONZE_ID = f"{PROJECT_ID}.{DATASET_ID}.{BRONZE_TABLE}"
FULL_SILVER_ID = f"{PROJECT_ID}.{DATASET_ID}.{SILVER_TABLE}"

client = bigquery.Client(project=PROJECT_ID)

print(f"Origem (Bronze): {FULL_BRONZE_ID}")
print(f"Destino (Silver): {FULL_SILVER_ID}")

## Inspeção rápida da Bronze

Só uma amostra, para conferir que a tabela existe e tem cara de VRA —
não carregamos a tabela inteira para pandas (ver nota acima).

In [ ]:
df_preview = client.query(
    f"SELECT * FROM `{FULL_BRONZE_ID}` LIMIT 5"
).to_dataframe()
df_preview

## Transformação Bronze → Silver

Pontos que a query abaixo resolve, descobertos ao inspecionar os 36
arquivos do recorte 2022-2024:

- **`dt_referencia` tem dois formatos na fonte** — `AAAA-MM-DD` nos
  arquivos mais antigos e `DD/MM/AAAA HH:MM:SS` nos mais novos (a mesma
  leva que trouxe a coluna `Codeshare`). A query tenta os dois formatos
  com `COALESCE`+`SAFE.PARSE_DATE`.
- **Campos vazios não podem estourar o parse** — todo `PARSE_DATETIME`/
  `PARSE_DATE` usa a variante `SAFE.`, que retorna `NULL` em vez de
  erro.
- **Chave de dedup**: companhia + número do voo + data de
  referência + aeroporto de origem + partida prevista — a identidade
  natural de uma perna de voo. Em empate, fica o registro com
  `_ingested_at` mais recente.
- **Faixa plausível de atraso**: `±1440 min` (±24h) — descarta
  exatamente os extremos (−1.453 e +3.894 min), que é atribuído a erro de virada de data, e preserva a população
  legítima de voos "Antecipado" (atraso negativo, ~52% da base).
- **Particionamento por `dt_referencia`**.
- **Clusterização** por companhia e aeroporto de origem
  — só é possível agora porque a Silver é tabela
  nativa (a Bronze, external table, não aceita `CLUSTER BY`).

In [ ]:
sql_silver = f"""
CREATE OR REPLACE TABLE `{FULL_SILVER_ID}`
PARTITION BY dt_referencia
CLUSTER BY sg_empresa_icao, sg_icao_origem
AS
WITH tipado AS (
  SELECT
    sg_empresa_icao,
    nm_empresa,
    nr_voo,
    cd_di,
    cd_tipo_linha,
    sg_equipamento_icao,
    SAFE_CAST(nr_assentos_ofertados AS INT64) AS nr_assentos_ofertados,
    sg_icao_origem,
    nm_aerodromo_origem,
    SAFE.PARSE_DATETIME('%d/%m/%Y %H:%M', dt_partida_prevista) AS dt_partida_prevista,
    SAFE.PARSE_DATETIME('%d/%m/%Y %H:%M', dt_partida_real)     AS dt_partida_real,
    sg_icao_destino,
    nm_aerodromo_destino,
    SAFE.PARSE_DATETIME('%d/%m/%Y %H:%M', dt_chegada_prevista) AS dt_chegada_prevista,
    SAFE.PARSE_DATETIME('%d/%m/%Y %H:%M', dt_chegada_real)     AS dt_chegada_real,
    ds_situacao_voo,
    NULLIF(ds_justificativa, '') AS ds_justificativa,
    COALESCE(
      SAFE.PARSE_DATE('%Y-%m-%d', dt_referencia),
      SAFE.PARSE_DATE('%d/%m/%Y', SUBSTR(dt_referencia, 1, 10))
    ) AS dt_referencia,
    ds_situacao_partida,
    ds_situacao_chegada,
    ds_codeshare,
    _ingested_at,
    _batch_id,
    _FILE_NAME AS _source_uri
  FROM `{FULL_BRONZE_ID}`
  WHERE ds_situacao_voo = 'REALIZADO'
),
deduplicado AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY sg_empresa_icao, nr_voo, dt_referencia, sg_icao_origem, dt_partida_prevista
      ORDER BY _ingested_at DESC
    ) AS rn
  FROM tipado
  QUALIFY rn = 1
)
SELECT
  * EXCEPT(rn),
  DATETIME_DIFF(dt_partida_real, dt_partida_prevista, MINUTE) AS atraso_partida_minutos
FROM deduplicado
WHERE
  dt_partida_prevista IS NOT NULL
  AND dt_partida_real IS NOT NULL
  AND ABS(DATETIME_DIFF(dt_partida_real, dt_partida_prevista, MINUTE)) <= 1440
"""

In [ ]:
job = client.query(sql_silver)
job.result()
print(f"Tabela Silver criada: {FULL_SILVER_ID}")
print(f"Bytes processados: {job.total_bytes_processed:,}")

## Validação

A contagem **cai** de Bronze para Silver — isso é esperado e é a evidência de que a Silver fez trabalho real
(filtro de `REALIZADO`, pares incompletos e outliers descartados).

In [ ]:
bronze_rows = client.get_table(FULL_BRONZE_ID).num_rows
silver_rows = client.get_table(FULL_SILVER_ID).num_rows
queda_pct = 100 * (1 - silver_rows / bronze_rows)

print(f"Linhas na Bronze: {bronze_rows:,}")
print(f"Linhas na Silver: {silver_rows:,}")
print(f"Queda: {queda_pct:.1f}%")

In [ ]:
df_silver_preview = client.query(
    f"SELECT * FROM `{FULL_SILVER_ID}` ORDER BY dt_referencia DESC LIMIT 5"
).to_dataframe()
df_silver_preview